In [407]:
import math
import numpy as np
import mip
from collections import namedtuple
import glob
import pandas as pd
from IPython.display import display, Markdown

In [408]:
##### Open the main sheet #####

directory = "examples/Mariners"
globFilename = directory + "/Full Name*.xls*"
excelFiles = glob.glob(globFilename)
if len(excelFiles) < 1 or len(excelFiles) > 1:
    print(f"Can't find unique excel file: {globFilename}")
    exit()
mainSheet = pd.read_excel(excelFiles[0])

In [409]:
##### Find the constraint row #####
for constraintRow in range(len(mainSheet.Date)):
    if mainSheet.iat[constraintRow,2] == "At least":
        break

In [410]:
##### Establish an object for each day of the season
 
Game = namedtuple('Game', ('weekday', 'month', 'day', 'gameDay', 'time', 'opponent', 'type', 'price', 'pairs', 'seats'))

##### Read the season schedule and store it as a list of games
 
openingDay = mainSheet.Date[0]
schedule = []
GamesInPlan = 0
PairsInPlan = 0
MaxPairsPerGame = 0
for weekday, date, time, opponent, type, price, pairs, seats in zip(mainSheet.Day, mainSheet.Date, mainSheet.Time, mainSheet.Opponent, mainSheet.Type, mainSheet.Price, mainSheet.GamePairs, mainSheet.Seats):
    if not isinstance(weekday, str) or weekday == "" or pd.isna(date):
        break
    schedule.append(Game(weekday, date.month, date.day, (date - openingDay).days, time.strftime("%I:%M %p"), opponent, type, price, pairs, seats))
    GamesInPlan += 1
    PairsInPlan += pairs
    MaxPairsPerGame = max(MaxPairsPerGame, pairs)


In [411]:
#########################################################
# Build a dictionary out of a list of games
#
#     buildCode == 0:    Create constraints for pair and quads
#     buildCode == 2:    Create constraint for pairs only
#     buildCode == 4:    Create constraint for quads only

def BuildDict(gameList, buildCode = 0):
    pairList = []
    for game in gameList:
        if buildCode != 4:
            pairList.append((game, 1.0))
        if buildCode != 2:
            pairList.append((game + GamesInPlan, 1.0))
    return dict(pairList)

# Define a class to contain constraints

class Constraint:
    def __init__(self, type, rhs, coefDict):
        self.type = type
        self.rhs = rhs
        self.coefDict = coefDict

# Define a class to contain each person's preferences

class SportsFan:
    def __init__(self, name, pairs, quads, ranking, extra):
        self.name = name
        self.pairs = pairs
        self.quads = quads

# Assign weights to games

        if len(ranking) != 0:
            self.ranking = ranking
        else:
            self.ranking = GamesInPlan * [GamesInPlan // 2]

# Adjust weights to favor highly ranked games

        self.useRanking = []
        midpoint = (GamesInPlan - 1) // 2
        for wgt in self.ranking:
            if wgt <= midpoint + 1:
                self.useRanking.append(math.sqrt(wgt - 1.0))
            else:
                self.useRanking.append(2.0 * math.sqrt(midpoint) - math.sqrt(2.0 * midpoint - wgt + 1))
        if len(self.useRanking) == GamesInPlan:
            self.useRanking += [2.0 * cost for cost in self.useRanking]
        else:
            for ix in range(GamesInPlan):
                self.useRanking[GamesInPlan + ix] *= 2

# Each person must attend correct number of games

        pairCon = Constraint('=', self.pairs, dict([(ix, 1.0) for ix in range(GamesInPlan)]))
        quadCon = Constraint('=', self.quads, dict([(ix, 1.0) for ix in range(GamesInPlan, 2 * GamesInPlan)]))

# Save all of the constraints for this person

        self.constraints = [pairCon, quadCon] + extra

##### This constraint handles the spacing of games

def Spacing(daysApart, comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    constraintList = []
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].gameDay + daysApart > schedule[lastGame].gameDay:
            lastGame += 1
        constraintList.append(Constraint(comparator, value, BuildDict(range(firstGame, lastGame), pairsOrQuads)))
        if lastGame == GamesInPlan:
            break
        while schedule[firstGame].gameDay + daysApart <= schedule[lastGame].gameDay:
            firstGame += 1
    return constraintList

##### Require or forbid games in various months

def Monthly(comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    constraintList = []
    while True:
        while schedule[lastGame].month < 5:
            lastGame += 1
        while lastGame < GamesInPlan and schedule[firstGame].month == schedule[lastGame].month:
            lastGame += 1
        constraintList.append(Constraint(comparator, value, BuildDict(range(firstGame, lastGame), pairsOrQuads)))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraintList

##### Require or forbid games in different series

def Series(comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    constraintList = []
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        constraintList.append(Constraint(comparator, value, BuildDict(range(firstGame, lastGame), pairsOrQuads)))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraintList

##### Require or forbid games for different opponents

def Opponents(comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    opponentDict = {}
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        opponentGames = opponentDict.get(schedule[firstGame].opponent, [])
        opponentGames += range(firstGame, lastGame)
        opponentDict[schedule[firstGame].opponent] = opponentGames
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    constraintList = []
    for opponentGames in opponentDict.values():
        constraintList.append(Constraint(comparator, value, BuildDict(opponentGames, pairsOrQuads)))
    return constraintList


In [412]:
##### Define the participants here #####

fans = []
totalPairs = 0
for fullName, nPairs, nQuads in zip(mainSheet.FullName, mainSheet.Pairs, mainSheet.Quads):
    if not isinstance(fullName, str) or fullName == "":
        break
    totalPairs += nPairs + 2 * nQuads
    globFilename = directory + "/" + fullName + "*.xls*"
    excelFiles = glob.glob(globFilename)
    if len(excelFiles) < 1 or len(excelFiles) > 1:
        print(f"Can't find unique excel file: {globFilename}")
        continue
    fanSheet = pd.read_excel(excelFiles[0])
    pairsRank = [np.int64(fanSheet.Pick[ix]) for ix in range(GamesInPlan)]
    if not isinstance(fanSheet.QuadPick[0], np.float64) and not math.isnan(fanSheet.QuadPick[0]):
        quadsRank = [np.int64(fanSheet.QuadPick[ix]) for ix in range(GamesInPlan)]
        pairsRank += quadsRank

    # Add fan constraints
    extraConstraints = []
    row = constraintRow
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Monthly('>', constraintValue)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Monthly('<', constraintValue)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Monthly('>', constraintValue, 2)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Monthly('<', constraintValue, 2)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Monthly('>', constraintValue, 4)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Monthly('<', constraintValue, 4)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Spacing(constraintValue + 1, '<', 1)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Spacing(constraintValue + 1, '>', 1)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Spacing(constraintValue + 1, '<', 1)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Spacing(constraintValue + 1, '>', 2)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Spacing(constraintValue + 1, '<', 4)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Spacing(constraintValue + 1, '>', 4)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Series('<', constraintValue)
    row += 1
    if fanSheet.iat[row,1]:
        constraintValue = fanSheet.iat[row, 3]
        extraConstraints += Opponents('<', constraintValue)
    fans.append(SportsFan(fullName, nPairs, nQuads, pairsRank, extraConstraints))

leftOver = PairsInPlan - totalPairs
if leftOver < 0:
    print("Too many games requested")
if leftOver > 0:
    pairsRank = (GamesInPlan * [np.int64(1)])[:]
    fans.append(SportsFan("Spare Pair", leftOver, 0, pairsRank, []))
    print(f"{leftOver} spare pairs")

73 spare pairs


In [413]:
for gix, game in enumerate(schedule):
    picks = []
    if gix < GamesInPlan:
        for fan in fans:
            picks.append(int(fan.ranking[gix]))
    print(f"{picks} {game.month}/{game.day} {game.weekday} {game.opponent}{game.time} ${game.price:,.2f}")

[80, 63, 14, 1] 3/26 Thu Guardians 07:10 PM $90.00
[50, 30, 73, 1] 3/27 Fri Guardians 06:45 PM $69.00
[69, 36, 71, 1] 3/28 Sat Guardians 06:40 PM $69.00
[45, 28, 64, 1] 3/29 Sun Guardians 04:20 PM $69.00
[7, 29, 41, 1] 3/30 Mon Yankees 06:40 PM $45.00
[65, 9, 60, 1] 3/31 Tue Yankees 06:40 PM $45.00
[8, 31, 9, 1] 4/1 Wed Yankees 01:10 PM $45.00
[51, 42, 22, 1] 4/10 Fri Astros 06:40 PM $45.00
[16, 19, 37, 1] 4/11 Sat Astros 06:40 PM $45.00
[52, 43, 68, 1] 4/12 Sun Astros 01:10 PM $45.00
[17, 21, 3, 1] 4/13 Mon Astros 01:10 PM $69.00
[34, 3, 67, 1] 4/17 Fri Rangers 06:40 PM $69.00
[40, 77, 16, 1] 4/18 Sat Rangers 04:15 PM $69.00
[20, 35, 46, 1] 4/19 Sun Rangers 01:10 PM $69.00
[28, 65, 58, 1] 4/20 Mon Athletics 06:40 PM $69.00
[2, 46, 78, 1] 4/21 Tue Athletics 06:40 PM $69.00
[29, 68, 27, 1] 4/22 Wed Athletics 01:10 PM $45.00
[61, 73, 51, 1] 5/1 Fri Royals 06:45 PM $45.00
[47, 56, 49, 1] 5/2 Sat Royals 06:40 PM $90.00
[71, 71, 81, 1] 5/3 Sun Royals 01:10 PM $90.00
[75, 22, 10, 1] 5/4 Mon 

In [414]:
try:
    tixModel = mip.Model()
    tixVars = [tixModel.add_var(var_type = mip.BINARY) for ix in range(2 * GamesInPlan * len(fans))]

    # All tickets must be allocated

    for ix in range(GamesInPlan):
        tixModel += mip.xsum(tixVars[ix + 2 * iy * GamesInPlan] + 2.0 * tixVars[ix + GamesInPlan + 2 * iy * GamesInPlan] for iy in range(len(fans))) == schedule[ix].pairs

    # Each fan must attend the correct number of games + satisfy all personal constraints

    for iy, fan in enumerate(fans):
        for constraint in fan.constraints:
            linfunc = mip.xsum(constraint.coefDict[ix] * tixVars[ix + iy * 2 * GamesInPlan] for ix in constraint.coefDict)
            if constraint.type == '=':
                tixModel += linfunc == constraint.rhs
            if constraint.type == '<':
                tixModel += linfunc <= constraint.rhs
            if constraint.type == '>':
                tixModel += linfunc >= constraint.rhs

    # Establish the objective function

    costs = []
    for fan in fans:
        costs += fan.useRanking
    tixModel.objective = mip.xsum(costs[ix] * tixVars[ix] for ix in range(len(tixVars)))
except Exception as e:
    print(f"Mip setup exception: {e}")

In [415]:
try:
    status = tixModel.optimize()
except Exception as e:
    print(f"Mip optimize exception: {e}")

if status != mip.OptimizationStatus.OPTIMAL:
    print(f"No solution found: {status}")

In [416]:
picks = [[] for fan in fans]
for mix, mvar in enumerate(tixModel.vars):
    if mvar.x is not None and mvar.x > 0.5:
        fix = mix // (2 * GamesInPlan)
        if len(fans[fix].ranking) > GamesInPlan:
            gix = mix % (2 * GamesInPlan)
        else:
            gix = mix % GamesInPlan
        picks[fix].append(fans[fix].ranking[gix])
for fix, fan in enumerate(fans):
    picks[fix].sort()
    picks[fix] = [int(pick) for pick in picks[fix]]
    print(fans[fix].name, picks[fix])

Tom Grandine [3, 7, 10, 11, 14, 18, 24, 25, 26, 29, 30, 54]
Al Erisman [4, 7, 9, 10, 12, 13, 15, 19, 20, 20, 23, 26, 27, 37, 38, 39, 41, 45, 53, 56, 58, 62]
Eric Brechner [1, 2, 5, 6, 7, 8, 9, 10, 12, 13, 14, 17, 19, 22, 23, 24, 28, 31, 45, 51, 63, 65]
Spare Pair [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [417]:
costs = (len(fans) * [0.0])[:]
gameAllocationTable = "Day | Date | Time | Opponent | Type | Seats |" + " |" * MaxPairsPerGame + "\n"
gameAllocationTable += "| :-: | :-: | :-: | :-: | :-: | :-: |"  + " :-: |" * MaxPairsPerGame + "\n"
for gix, game in enumerate(schedule):
    gameAllocationTable += f"| {game.weekday} | {game.month}/{game.day} | {game.time} | {game.opponent} | {game.type} | {game.seats} |"
    for mix, mvar in enumerate(tixModel.vars):
        if mix % GamesInPlan == gix and mvar.x is not None and mvar.x > 0.5:
            fix = mix // (2 * GamesInPlan)
            if len(fans[fix].ranking) > GamesInPlan and mix % (2 * GamesInPlan) >= GamesInPlan:
                gix += GamesInPlan
            costs[fix] += 2.0 * game.price
            gameAllocationTable += f" {fans[fix].name} ({fans[fix].ranking[gix]}) |"
            if mix % (2 * GamesInPlan) >= GamesInPlan:
                costs[fix] += 2.0 * game.price
                gameAllocationTable += " |"
    gameAllocationTable += " ❌ |" * (MaxPairsPerGame - game.pairs) + "\n"
display(Markdown(gameAllocationTable))

Day | Date | Time | Opponent | Type | Seats | | |
| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| Thu | 3/26 | 07:10 PM | Guardians  | A | $90.00 (4) | Eric Brechner (14) | Spare Pair (1) |
| Fri | 3/27 | 06:45 PM | Guardians  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sat | 3/28 | 06:40 PM | Guardians  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sun | 3/29 | 04:20 PM | Guardians  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Mon | 3/30 | 06:40 PM | Yankees  | D | $45.00 (4) | Tom Grandine (7) | |
| Tue | 3/31 | 06:40 PM | Yankees  | D | $45.00 (4) | Al Erisman (9) | Spare Pair (1) |
| Wed | 4/1 | 01:10 PM | Yankees  | D | $45.00 (4) | Eric Brechner (9) | Spare Pair (1) |
| Fri | 4/10 | 06:40 PM | Astros  | D | $45.00 (4) | Eric Brechner (22) | Spare Pair (1) |
| Sat | 4/11 | 06:40 PM | Astros  | D | $45.00 (4) | Al Erisman (19) | Spare Pair (1) |
| Sun | 4/12 | 01:10 PM | Astros  | D | $45.00 (4) | Al Erisman (10) | |
| Mon | 4/13 | 01:10 PM | Astros  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Fri | 4/17 | 06:40 PM | Rangers  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sat | 4/18 | 04:15 PM | Rangers  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sun | 4/19 | 01:10 PM | Rangers  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Mon | 4/20 | 06:40 PM | Athletics  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Tue | 4/21 | 06:40 PM | Athletics  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Wed | 4/22 | 01:10 PM | Athletics  | D | $45.00 (4) | Tom Grandine (29) | Spare Pair (1) |
| Fri | 5/1 | 06:45 PM | Royals  | D | $45.00 (4) | Eric Brechner (51) | Spare Pair (1) |
| Sat | 5/2 | 06:40 PM | Royals  | A | $90.00 (4) | Al Erisman (56) | Spare Pair (1) |
| Sun | 5/3 | 01:10 PM | Royals  | A | $90.00 (4) | Al Erisman (20) | |
| Mon | 5/4 | 06:40 PM | Braves  | A | $90.00 (4) | Eric Brechner (10) | Spare Pair (1) |
| Tue | 5/5 | 06:40 PM | Braves  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Wed | 5/6 | 01:10 PM | Braves  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Fri | 5/15 | 06:40 PM | Padres  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sat | 5/16 | 04:15 PM | Padres  | D | $45.00 (4) | Tom Grandine (54) | Spare Pair (1) |
| Sun | 5/17 | 04:20 PM | Padres  | D | $45.00 (4) | Eric Brechner (65) | Spare Pair (1) |
| Mon | 5/18 | 06:40 PM | White Sox  | D | $45.00 (4) | Al Erisman (7) | Spare Pair (1) |
| Tue | 5/19 | 06:40 PM | White Sox  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Wed | 5/20 | 01:10 PM | White Sox  | B | $69.00 (2) | Eric Brechner (28) | ❌ |
| Fri | 5/29 | 07:10 PM | D-backs  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sat | 5/30 | 07:10 PM | D-backs  | C | $57.00 (4) | Tom Grandine (11) | Spare Pair (1) |
| Sun | 5/31 | 01:10 PM | D-backs  | C | $57.00 (4) | Eric Brechner (2) | Spare Pair (1) |
| Mon | 6/1 | 06:40 PM | Mets  | C | $57.00 (4) | Eric Brechner (12) | Spare Pair (1) |
| Tue | 6/2 | 06:40 PM | Mets  | A | $90.00 (4) | Al Erisman (39) | Spare Pair (1) |
| Wed | 6/3 | 12:40 PM | Mets  | A | $90.00 (4) | Tom Grandine (26) | Spare Pair (1) |
| Tue | 6/16 | 06:40 PM | Orioles  | A | $90.00 (4) | Eric Brechner (63) | Spare Pair (1) |
| Wed | 6/17 | 06:40 PM | Orioles  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Thu | 6/18 | 01:10 PM | Orioles  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Fri | 6/19 | 07:10 PM | Red Sox  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sat | 6/20 | 07:10 PM | Red Sox  | C | $57.00 (4) | Al Erisman (37) | Spare Pair (1) |
| Sun | 6/21 | 01:10 PM | Red Sox  | C | $57.00 (4) | Eric Brechner (24) | Spare Pair (1) |
| Mon | 6/29 | 06:40 PM | Angels  | C | $57.00 (4) | Al Erisman (15) | Spare Pair (1) |
| Tue | 6/30 | 06:40 PM | Angels  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Thu | 7/2 | 06:40 PM | Angels  | A | $90.00 (4) | Eric Brechner (8) | Spare Pair (1) |
| Fri | 7/3 | 07:10 PM | Blue Jays  | A | $90.00 (4) | Eric Brechner (6) | Spare Pair (1) |
| Sat | 7/4 | 01:10 PM | Blue Jays  | A | $90.00 (4) | Tom Grandine (24) | Spare Pair (1) |
| Sun | 7/5 | 02:00 PM | Blue Jays  | A | $90.00 (4) | Al Erisman (41) | Spare Pair (1) |
| Fri | 7/17 | 07:10 PM | Giants  | A | $90.00 (4) | Al Erisman (13) | Spare Pair (1) |
| Sat | 7/18 | 05:08 PM | Giants  | A | $90.00 (4) | Eric Brechner (19) | Spare Pair (1) |
| Sun | 7/19 | 01:10 PM | Giants  | C | $57.00 (4) | Al Erisman (23) | Spare Pair (1) |
| Mon | 7/20 | 06:40 PM | Reds  | C | $57.00 (4) | Eric Brechner (45) | Spare Pair (1) |
| Tue | 7/21 | 06:40 PM | Reds  | C | $57.00 (4) | Tom Grandine (18) | Spare Pair (1) |
| Wed | 7/22 | 12:40 PM | Reds  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Fri | 7/31 | 07:10 PM | Twins  | A | $90.00 (4) | Eric Brechner (7) | Spare Pair (1) |
| Sat | 8/1 | 01:10 PM | Twins  | A | $90.00 (4) | Tom Grandine (30) | Spare Pair (1) |
| Sun | 8/2 | 01:10 PM | Twins  | A | $90.00 (4) | Tom Grandine (3) | |
| Tue | 8/4 | 06:40 PM | Tigers  | C | $57.00 (4) | Eric Brechner (23) | Spare Pair (1) |
| Wed | 8/5 | 06:40 PM | Tigers  | C | $57.00 (4) | Al Erisman (62) | Spare Pair (1) |
| Thu | 8/6 | 01:10 PM | Tigers  | C | $57.00 (4) | Al Erisman (26) | Spare Pair (1) |
| Fri | 8/7 | 07:10 PM | Rays  | A | $90.00 (4) | Tom Grandine (25) | Spare Pair (1) |
| Sat | 8/8 | 06:50 PM | Rays  | A | $90.00 (4) | Eric Brechner (17) | Spare Pair (1) |
| Sun | 8/9 | 01:10 PM | Rays  | A | $90.00 (4) | Al Erisman (58) | Spare Pair (1) |
| Fri | 8/21 | 07:10 PM | Cubs  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sat | 8/22 | 06:40 PM | Cubs  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Sun | 8/23 | 01:10 PM | Cubs  | B | $69.00 (2) | Eric Brechner (13) | ❌ |
| Mon | 8/24 | 06:40 PM | Phillies  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Tue | 8/25 | 06:40 PM | Phillies  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Wed | 8/26 | 01:10 PM | Phillies  | B | $69.00 (2) | Eric Brechner (1) | ❌ |
| Thu | 9/3 | 06:40 PM | Athletics  | C | $57.00 (4) | Tom Grandine (14) | Spare Pair (1) |
| Fri | 9/4 | 07:10 PM | Athletics  | C | $57.00 (4) | Al Erisman (53) | Spare Pair (1) |
| Sat | 9/5 | 06:40 PM | Athletics  | C | $57.00 (4) | Eric Brechner (31) | Spare Pair (1) |
| Sun | 9/6 | 01:10 PM | Athletics  | C | $57.00 (4) | Al Erisman (4) | Spare Pair (1) |
| Tue | 9/8 | 06:40 PM | Rangers  | B | $69.00 (2) | Eric Brechner (5) | ❌ |
| Wed | 9/9 | 06:40 PM | Rangers  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Thu | 9/10 | 01:10 PM | Rangers  | B | $69.00 (2) | Spare Pair (1) | ❌ |
| Tue | 9/22 | 06:40 PM | Astros  | C | $57.00 (4) | Tom Grandine (10) | Spare Pair (1) |
| Wed | 9/23 | 07:10 PM | Astros  | C | $57.00 (4) | Al Erisman (45) | Spare Pair (1) |
| Thu | 9/24 | 06:40 PM | Angels  | C | $57.00 (4) | Al Erisman (27) | Spare Pair (1) |
| Fri | 9/25 | 07:10 PM | Angels  | A | $90.00 (4) | Al Erisman (38) | Spare Pair (1) |
| Sat | 9/26 | 06:40 PM | Angels  | A | $90.00 (4) | Al Erisman (20) | Spare Pair (1) |
| Sun | 9/27 | 12:10 PM | Angels  | A | $90.00 (4) | Al Erisman (12) | Spare Pair (1) |


In [418]:
amountOwedTable = "| | Amount owed | Paid |\n"
amountOwedTable += "| :- | -: | -: |\n"
for fix, fan in enumerate(fans):
    amountOwedTable += f"| {fan.name} | ${costs[fix]:,.2f} | |\n"
display(Markdown(amountOwedTable))

| | Amount owed | Paid |
| :- | -: | -: |
| Tom Grandine | $1,896.00 | |
| Al Erisman | $3,276.00 | |
| Eric Brechner | $3,036.00 | |
| Spare Pair | $10,026.00 | |
